In [ ]:
import shutil
from pathlib import Path

import s3fs
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from byte_util.met_transport import write_transient, create_bcvs_soi
from byte_util import all_sites

rerun_bcvs = False
rerun_infile = False

s3 = s3fs.S3FileSystem(anon=False)
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'

sim_types = ['spinup', 'hourly', 'daily', 'monthly']

## First, select additional dates at which to start simulations

In [ ]:
from scipy.stats.qmc import LatinHypercube
sampler = LatinHypercube(d=1, seed=1)

# Generate 39 additional start dates
samples = sampler.random(n=39) * 3650  # In days since 2001-01-01, up to 2010-12-30
samples = np.round(samples).astype(int)

start_dates_idx = pd.to_datetime('2001-01-01') + pd.to_timedelta(np.squeeze(samples), 'D')
start_dates = [s.strftime('%Y-%m-%d') for s in start_dates_idx]  # Convert to str
start_dates.sort()

## Next, create directory structure for all simulations

In [ ]:
basepath = Path('../simulations/met_forcing_transport/min3p_runs_variable_start')
for start_date in start_dates:
    if not rerun_bcvs or not rerun_infile:
        continue
    for site in all_sites:
        # Create run directories
        sitepath = basepath / f'{start_date}/{site}'
        for sim_type in sim_types:
            simpath = sitepath / sim_type
            shutil.rmtree(simpath, ignore_errors=True)
            simpath.mkdir(parents=True)

## Next, create bcvs and soi files for each simulation

In [ ]:
# Create hourly/daily/monthly bcvs and soi files
base_path = Path('../simulations/met_forcing_transport/min3p_runs/base/')

# A dict of the name we'll call this frequency and the corresponding pandas resampling string
all_frequencies = {'hourly': 'h',
                   'daily': 'D',
                   'monthly': 'MS'}

for site in tqdm(all_sites):
    if not rerun_bcvs:
        continue
    forcing_file = f'{s3_base_path}/input-data/processed-data/climate/{site}_hourly_forcing.csv'
    met_forcing = pd.read_csv(forcing_file, parse_dates=True, index_col=0)

    # Convert from mm/hr to m/s and m/d
    met_forcing['surface_flux_m.s'] = met_forcing['surface_flux_mm.hr'] / 1000 / 60 / 60
    met_forcing['transpiration_m.d'] = met_forcing['transpiration_mm.hr'] / 1000 * 24

    for start_date in start_dates:
        for freq_name, freq in all_frequencies.items():
            # Setting delete_first_record to false and dt to 0.001 means that the
            # first timestamp in the bcvs and soi files will be 0.001 d. As a result,
            # the BC and transpiration values entered into the main input file will
            # quickly be overwritten
            bcvs, soi = create_bcvs_soi(met_forcing, freq=freq, start_date=start_date,
                                        delete_first_record=False, dt=1e-3)

            # Limit to first ~6 years to save space
            mask = soi['time'] < 2191
            soi = soi.loc[mask, :]
            bcvs = bcvs.loc[mask, :]

            # Export to file
            simpath = basepath / start_date / site / freq_name
            write_transient(simpath / f'{freq_name}.bcvs', bcvs)
            write_transient(simpath / f'{freq_name}.soi', soi)

In [ ]:
## Create main MIN3P input file (*.dat)
from min3p.input import InputFile

for site in tqdm(all_sites):
    if not rerun_infile:
        continue

    spinup_path = Path(f'../simulations/met_forcing_transport/min3p_runs/{site}/spinup')
    infile = InputFile.load('spinup.dat', path=spinup_path)
    for sim_type in sim_types:

        # Update database directories
        infile.geochemical_system.database_directory = '../../../databases'

        ## Adjust scenario-specific parameters
        if sim_type == 'spinup':
            # Adjust simulation end time
            infile.time_step_control.final_time = 3650.
        else:
            # Adjust simulation end time
            infile.time_step_control.final_time = 2190.

            # Adjust output times for spatial (contour) data
            oc = infile.output_control
            output_times = [float(t) for t in range(365, 2191, 365)]
            oc.output_of_spatial_data = output_times

            # Adjust initial conditions
            infile.initial_conditions_vsflow.text = ("! Initial cond generated from spin-up\n"
                                                     "'read initial condition from file'\n")

            # Adjust initial conditions to include tracers
            icrt = infile.initial_conditions_reactive_transport
            background = icrt.zones[0]
            background.extent_of_zone = [0.0, 1.0, 0.0, 1.0, 0.0, 3.0]
            deep_tracer = ("\n'concentration input'\n\n"
                           "1.0000d-3      'free'         ;'tracer - psi01'\n"
                           "1.0000d-1      'free'         ;'tracer - psi02'\n\n"
                           "'extent of zone'\n"
                           "0.0 1.0  0.0 1.0  3.0 3.7\n"
            )
            _ = icrt.add_zone(name='deep tracer', body=deep_tracer)

            shallow_tracer = ("\n'concentration input'\n\n"
                              "1.0000d-1      'free'         ;'tracer - psi01'\n"
                              "1.0000d-3      'free'         ;'tracer - psi02'\n\n"
                              "'extent of zone'\n"
                              "0.0 1.0  0.0 1.0  3.7 4.0\n"
            )
            _ = icrt.add_zone(name='shallow tracer', body=shallow_tracer)

            # Adjust concentration of top BC to 1e-3
            bcrt = infile.boundary_conditions_reactive_transport  # Get BC block
            top_conc = bcrt.zones[0].concentration_input  # Get concentrations of top BC
            top_conc.records[0].replace_content("1.0000d-3      'free'")  # Adjust 1st comp (tracer)

            # If sim != longterm or spinup, add transient keywords
            if sim_type != 'longterm':
                bcvs = infile.boundary_conditions_vsflow
                bcvs.text += "\n'transient boundary conditions'\n\n"
                infile.physical_parameters_vsflow.transient_transpiration = True

        for start_date in start_dates:
            # Save to file
            simpath = basepath / start_date / site / sim_type
            sim_file = simpath / f'{sim_type}.dat'
            infile.save(sim_file)

            # Also copy over *rld file
            shutil.copy(spinup_path / 'spinup.rld', simpath / f'{sim_type}.rld')

In [ ]:
# Write all the start dates to a run list file for use in the SLURM job array
with open('../simulations/met_forcing_transport/min3p_runs_variable_start/start_dates.txt', 'w') as f:
    for start_date in start_dates:
        f.write(f'{start_date}\n')

# Also copy databases directory to the new simulation directory
if rerun_infile or rerun_bcvs:
    database_dir = '../simulations/met_forcing_transport/min3p_runs_variable_start/databases'
    shutil.rmtree(database_dir, ignore_errors=True)
    shutil.copytree('../simulations/databases', database_dir, dirs_exist_ok=True)

In [ ]:
# Finally, tar all the simulation directories into a single archive
import tarfile

if rerun_infile or rerun_bcvs:
    with tarfile.open('../simulations/met_forcing_transport/min3p_runs_variable_start.tar.gz', 'w:gz') as tar:
        # Add sbatch and list of simulations to run
        sbatch_file = 'met_transport_variable_start.sbatch'
        tar.add(basepath / sbatch_file, arcname=f'{basepath.name}/{sbatch_file}')
        tar.add(basepath / 'start_dates.txt', arcname=f'{basepath.name}/start_dates.txt')

        # Add databases to tar and remove original
        tar.add(database_dir, arcname=f'{basepath.name}/databases')
        shutil.rmtree(database_dir, ignore_errors=True)

        # Add each start date to tar and remove original
        for start_date in start_dates:
            tar.add(basepath / start_date, arcname=f'{basepath.name}/{start_date}')
            shutil.rmtree(basepath / start_date, ignore_errors=True)